# 데이터 정렬과 유용한 메서드 정답지

이 노트북은 `Pandas_4_arrange_method_Practice.ipynb`의 모든 연습문제(1 ~ 14번)에 대한 **모범 풀이 코드와 결과 해석**을 수록한 정답지입니다.
단일/다중 정렬(`sort_values`), 인덱스 정렬(`sort_index`), `apply` vs 벡터 연산 성능, 문자열 처리(`str.split`), `dateutil.rrule`과 `pd.date_range` 시계열 생성을 완벽히 다룹니다.


## 처음 배우는 분을 위한 안내

오름차순은 작은 값부터, 내림차순은 큰 값부터입니다. sort_values는 값으로 정렬하고 sort_index는 행 이름표로 정렬합니다. apply는 각 값에 같은 함수를 적용하고 lambda는 짧은 이름 없는 함수를 뜻합니다. 벡터 연산은 열 전체를 한 번에 계산하는 방법입니다.

**학습 순서:** 문제 1 ~ 6은 필수 기초, 함수·문자열·날짜를 다루는 7 ~ 14는 선택 도전입니다. rrule은 반복 날짜 규칙이며 12 ~ 13번에만 필요합니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

## 실행 준비

Python 3와 pandas가 필요합니다. 정렬 노트북의 rrule에는 python-dateutil, EDA의 선택 그래프에는 matplotlib이 필요합니다. 없는 경우 별도 코드 셀에서 `%pip install pandas python-dateutil matplotlib`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 생성됩니다.

Colab에서는 `.ipynb`를 열고 준비 셀부터 실행하세요. 아래 코드는 실습 데이터 생성만 수행합니다.

In [ ]:
import pandas as pd

# 학습용 가상 주문 데이터. 실제 고객 데이터가 아닙니다.
df = pd.DataFrame({
    'order_id': list(range(101, 113)),
    'order_date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
                   '2024-01-05', '2024-01-06', '2024-01-07', '2024-02-01',
                   '2024-02-02', '2024-02-03', '2024-02-04', '2024-02-05'],
    'customer_id': [1, 2, 1, 3, 4, 2, 5, 1, 3, 4, 5, 2],
    'product_category': ['Food', 'Book', 'Food', 'Tech', 'Book', 'Tech',
                         'Food', 'Tech', 'Book', 'Food', 'Tech', 'Book'],
    'region': ['South', 'North', 'South', 'East', 'North', 'South',
               'East', 'North', 'South', 'East', 'North', 'South'],
    'quantity': [2, 1, 3, 1, 2, 2, 1, 1, 3, 2, 1, 2],
    'unit_price': [100, 200, 100, 1000, 200, 1000, 100, 1000, 200, 100, 1000, 200],
    'discount': [0, 0.1, 0, 0.2, 0, 0.1, 0, 0, 0.1, 0, 0.2, 0],
    'payment_method': ['Card', 'Wallet', 'Card', 'COD', 'Card', 'Wallet',
                       'COD', 'Card', 'Wallet', 'Card', 'COD', 'Card'],
    'delivery_days': [1, 4, 3, 7, 2, 5, 8, 2, 6, 3, 9, 4],
    'customer_rating': [4.5, 4.0, 4.8, 2.5, 4.2, 3.5, 2.0, 4.6, 3.0, 4.4, 2.2, 4.1],
    'revenue': [200, 180, 300, 800, 400, 1800, 100, 1000, 540, 200, 800, 400]
})
df['order_date'] = pd.to_datetime(df['order_date'])
df_original = df.copy(deep=True)

customer_master = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 6],
    'membership_tier': ['Gold', 'Silver', 'Gold', 'Bronze', 'Bronze'],
    'region': ['South', 'North', 'East', 'North', 'West']
})
customer_summary = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'total_orders': [3, 3, 2, 2, 2],
    'total_spent': [1500, 2380, 1340, 600, 900],
    'avg_order_value': [500, 2380 / 3, 670, 300, 450]
})

codes = pd.DataFrame({'customer_id': ['CUST-001', 'CUST-002', 'CUST-003']})
date_text = pd.Series(['01/15/2024', '02/03/2024', '12/31/2024'], name='date_text')

## 문제 1. 매출 상위 주문 [필수 기초]

df에서 revenue 내림차순·order_id 오름차순으로 정렬해 상위 3건을 확인하세요.

**점검 기준:** 주문 106·108·104 순서입니다. 동점 800에서 ID가 작은 104가 먼저입니다.

In [ ]:
# 문제 1 풀이
sorted_rev = df.sort_values(by=['revenue', 'order_id'], ascending=[False, True])
top3 = sorted_rev.head(3)

display(top3[['order_id', 'revenue', 'product_category', 'region']])
print("상위 3건 order_id 순서:", top3['order_id'].tolist())


**결과 해석 / 확인한 점:**

1. 매출(`revenue`) 내림차순 및 동률 시 주문 ID(`order_id`) 오름차순으로 정렬한 상위 3건은 **106번(1,800원), 108번(1,000원), 104번(800원)** 순서입니다.
2. 주문 104번과 111번은 모두 매출 800원으로 동률이지만, 두 번째 정렬 기준인 `order_id` 오름차순에 따라 ID가 더 작은 104번이 111번보다 먼저 위치함을 확인했습니다.


## 문제 2. 배송이 오래 걸린 순서 [필수 기초]

배송일수 내림차순·주문 ID 오름차순으로 정렬하고 order_id·delivery_days·region의 앞 5행을 선택하세요.

**점검 기준:** 첫 주문은 111이며 배송일수는 9입니다. 정렬 전후 행 수는 같습니다.

In [ ]:
# 문제 2 풀이
sorted_deliv = df.sort_values(by=['delivery_days', 'order_id'], ascending=[False, True])
top5_deliv = sorted_deliv[['order_id', 'delivery_days', 'region']].head(5)

display(top5_deliv)
print(f"\n정렬 전 행 수: {len(df)}행, 정렬 후 행 수: {len(sorted_deliv)}행")
print(f"최장 배송 주문: {top5_deliv.iloc[0]['order_id']}번 ({top5_deliv.iloc[0]['delivery_days']}일 소요)")


**결과 해석 / 확인한 점:**

1. 배송일수 내림차순 상위 5건은 **111번(9일, North), 107번(8일, East), 104번(7일, East), 109번(6일, South), 106번(5일, South)** 순서입니다.
2. 정렬은 행의 순서만 재배열할 뿐 행의 추가나 삭제가 일어나지 않으므로, 정렬 전후 행 수는 **12행**으로 동일하게 보존됩니다.


## 문제 3. 평점 오름차순 [필수 기초]

고객 평점이 낮은 순으로 정렬하고 인덱스를 새로 부여하세요. 원래 df는 유지하세요.

**점검 기준:** 가장 낮은 평점은 주문 107의 2.0입니다. 원본 인덱스가 추가 열로 생기지 않도록 하세요.

In [ ]:
# 문제 3 풀이
sorted_rating = df.sort_values(by='customer_rating', ascending=True).reset_index(drop=True)

display(sorted_rating[['order_id', 'customer_rating', 'product_category']].head(3))
print(f"\n가장 낮은 평점: {sorted_rating.loc[0, 'customer_rating']}점 (주문 {sorted_rating.loc[0, 'order_id']}번)")
print("원래 df 인덱스 0의 order_id 보존 여부:", df.loc[0, 'order_id'] == 101)


**결과 해석 / 확인한 점:**

1. 고객 평점 오름차순 정렬 결과 가장 낮은 평점은 **주문 107번의 2.0점**(그 다음 111번 2.2점, 104번 2.5점)입니다.
2. `reset_index(drop=True)`를 사용하여 기존의 흐트러진 인덱스를 버리고 0부터 11까지 깔끔하게 새 인덱스를 부여했으며, 기존 인덱스가 불필요한 'index' 컬럼으로 추가되지 않도록 처리했습니다.
3. 원본 `df`의 첫 행은 여전히 101번 주문을 가리키고 있어 원본 데이터가 변경되지 않았음을 확인했습니다.


## 문제 4. 복수 기준 정렬 [필수 기초]

region 오름차순, revenue 내림차순, order_id 오름차순으로 정렬하세요. 매출 하나로 정렬한 결과와 목적이 어떻게 다른지 적으세요.

**점검 기준:** 지역이 먼저 모이고 그 안에서 매출 순서가 정해집니다. ascending의 순서가 by와 대응돼야 합니다.

In [ ]:
# 문제 4 풀이
sorted_multi = df.sort_values(
    by=['region', 'revenue', 'order_id'],
    ascending=[True, False, True]
)

display(sorted_multi[['region', 'revenue', 'order_id', 'product_category']])


**결과 해석 / 확인한 점:**

1. `by` 리스트와 `ascending` 리스트를 1:1로 대응시켜 지역(East -> North -> South)이 먼저 알파벳순으로 모이고, 각 지역 내에서 매출이 높은 순서로 정렬된 후, 매출 동률 시 주문 ID 순으로 정렬되었습니다.
2. **매출 하나로만 정렬한 결과와의 목적 차이:**
   - 단일 매출 정렬은 전체 비즈니스 관점에서 가장 큰 매출을 일으킨 주문을 식별하는 데 목적이 있습니다.
   - 복수 기준 정렬(지역별 매출 정렬)은 각 권역별(지사별) 1위 주문이나 지역 내 매출 기여 순위를 비교·평가하기 위한 그룹 내 서열화에 목적이 있습니다.


## 문제 5. loc와 iloc 비교 [필수 기초]

매출 내림차순으로 정렬하되 인덱스를 보존하세요. iloc[0]과 loc[0]의 order_id를 비교한 후 reset_index(drop=True)를 적용하세요.

**점검 기준:** 새 순번 전에는 106과 101입니다. 새 순번 후 두 선택의 첫 행이 같아집니다.

In [ ]:
# 문제 5 풀이
df_sorted_idx = df.sort_values(by='revenue', ascending=False)

print("=== 인덱스 재부여 전 ===")
print("iloc[0]['order_id'] (0번째 위치) :", df_sorted_idx.iloc[0]['order_id'])
print("loc[0]['order_id']  (레이블 0번)  :", df_sorted_idx.loc[0]['order_id'])

# reset_index(drop=True) 적용
df_reset = df_sorted_idx.reset_index(drop=True)

print("\n=== reset_index(drop=True) 적용 후 ===")
print("iloc[0]['order_id'] :", df_reset.iloc[0]['order_id'])
print("loc[0]['order_id']  :", df_reset.loc[0]['order_id'])
print("두 선택의 일치 여부:", df_reset.iloc[0]['order_id'] == df_reset.loc[0]['order_id'])


**결과 해석 / 확인한 점:**

1. **정렬 직후의 차이:**
   - 정렬 후 원본 인덱스가 그대로 유지되므로, `iloc[0]`은 정렬된 첫 번째 물리적 행인 **106번 주문(1,800원)**을 가리킵니다.
   - 반면 `loc[0]`은 원래 인덱스 레이블이 '0'이었던 행을 찾아가므로 정렬 위치와 무관하게 **101번 주문(200원)**을 가리킵니다.
2. `reset_index(drop=True)`를 적용한 후에는 레이블 인덱스가 물리적 행 순서와 일치하도록 0번부터 다시 매겨지므로, `iloc[0]`과 `loc[0]` 모두 동일하게 106번 주문을 가리키게 됩니다.


## 문제 6. 인덱스 기준 정렬 [필수 기초]

지역별 매출 합계 Series를 만든 뒤 지역 이름순과 매출 내림차순으로 각각 정렬하세요.

**점검 기준:** 이름순은 East·North·South입니다. 값 정렬은 South·North·East입니다.

In [ ]:
# 문제 6 풀이
region_rev = df.groupby('region')['revenue'].sum()

print("=== 원본 Series ===")
print(region_rev)

print("\n=== 지역 이름순 정렬 (sort_index, 오름차순) ===")
by_name = region_rev.sort_index(ascending=True)
print(by_name)

print("\n=== 매출 금액순 정렬 (sort_values, 내림차순) ===")
by_value = region_rev.sort_values(ascending=False)
print(by_value)


**결과 해석 / 확인한 점:**

1. **인덱스 기준 정렬(`sort_index()`):** 인덱스인 지역 레이블을 알파벳 오름차순으로 정렬하여 **East(1,100원) -> North(2,380원) -> South(3,240원)** 순으로 나열됩니다.
2. **값 기준 정렬(`sort_values(ascending=False)`):** 데이터 값인 매출 합계의 크기순으로 정렬하여 **South(3,240원) -> North(2,380원) -> East(1,100원)** 순으로 나열됩니다.


## 문제 7. apply와 lambda 산술 [선택 도전]

df의 복사본에 매출×1.1인 revenue_adjusted를 apply와 lambda로 만드세요. 벡터 연산 결과와 비교하세요.

**점검 기준:** 첫 행은 220입니다. 부동소수 오차를 고려해 비교하고 원래 revenue는 유지하세요. 1.1은 단순 실습 배율입니다.

In [ ]:
# 문제 7 풀이
df_calc = df.copy()

# apply와 lambda 사용
df_calc['revenue_adjusted'] = df_calc['revenue'].apply(lambda x: x * 1.1)

# 벡터 연산 결과와 비교
vector_res = df_calc['revenue'] * 1.1

# 부동소수점 오차 허용 비교
is_close = (abs(df_calc['revenue_adjusted'] - vector_res) < 1e-9).all()

display(df_calc[['order_id', 'revenue', 'revenue_adjusted']].head(3))
print(f"\n첫 번째 행 조정 매출: {df_calc.loc[0, 'revenue_adjusted']}")
print("apply 연산과 벡터 연산 결과 일치 여부:", is_close)


**결과 해석 / 확인한 점:**

1. 첫 행의 원본 매출 200원에 1.1을 곱한 `revenue_adjusted`는 **220.0원**으로 점검 기준과 정확히 일치합니다.
2. `apply(lambda ...)`와 판다스 내장 벡터 연산(`df['revenue'] * 1.1`)은 수학적으로 동일한 결과를 산출합니다.
3. **성능 관점의 시사점:** `apply`는 내부적으로 파이썬 인터프리터 수준에서 한 행씩 루프를 돌지만, 벡터 연산은 C/Cython 레벨에서 최적화된 SIMD 명령어로 처리되므로 수만 ~ 수백만 건의 대용량 데이터에서는 벡터 연산이 수십 ~ 수백 배 이상 빠릅니다.


## 문제 8. 할인율 백분율 [선택 도전]

discount를 100배해 discount_percent를 만드세요. 숫자 10과 표시 문자열 "10%"가 계산에서 어떤 차이가 있는지 적으세요.

**점검 기준:** 두 번째 행은 10입니다. 수치 계산용 열을 보고서 문자열로 덮어쓰지 마세요.

In [ ]:
# 문제 8 풀이
df_calc['discount_percent'] = df_calc['discount'] * 100

display(df_calc[['order_id', 'discount', 'discount_percent']].head(3))
print(f"\n두 번째 행 discount_percent 값: {df_calc.loc[1, 'discount_percent']} (자료형: {type(df_calc.loc[1, 'discount_percent'])})")


**결과 해석 / 확인한 점:**

1. `df_calc['discount'] * 100`을 통해 두 번째 행의 할인율 0.1이 **10.0(float64)**으로 계산되었습니다.
2. **숫자 `10` vs 표시 문자열 `"10%"`의 계산상 차이:**
   - 숫자 `10`은 수치형(numeric) 자료형이므로 평균, 최대, 최소 계산이나 조건 필터링(`discount_percent >= 10`)에 바로 활용할 수 있습니다.
   - 반면 `"10%"`는 문자열(string) 자료형이므로 수치 연산 시 `TypeError`가 발생하며, '2%'가 '10%'보다 사전순으로 뒤에 오는 등 잘못된 정렬을 유발합니다. 따라서 계산용 열은 수치형을 유지하고 포맷팅은 최종 화면 출력 직전에만 적용해야 합니다.


## 문제 9. 조건과 결측값 분류 [선택 도전]

별도 Series [3, 4, None]을 만들고 3 이하는 Fast, 초과는 Standard, 결측은 Unknown으로 분류하세요. 이후 df의 배송일수에도 적용하세요.

**점검 기준:** 경계값 3, 초과값 4, 누락값의 세 경우를 모두 점검합니다.

In [ ]:
# 문제 9 풀이
test_s = pd.Series([3, 4, None], name='sample_days')

def classify_delivery(days):
    if pd.isna(days):
        return 'Unknown'
    elif days <= 3:
        return 'Fast'
    else:
        return 'Standard'

res_test = test_s.apply(classify_delivery)
print("=== 테스트 Series 분류 결과 ===")
print(pd.DataFrame({'days': test_s, 'class': res_test}))

# df의 배송일수에 적용
df_calc['delivery_class'] = df_calc['delivery_days'].apply(classify_delivery)
print("\n=== df 적용 후 배송 분류 빈도 ===")
print(df_calc['delivery_class'].value_counts())


**결과 해석 / 확인한 점:**

1. **테스트 Series 검증:** 경계값 3은 `Fast`, 초과값 4는 `Standard`, 결측치 `None`은 `Unknown`으로 의도한 세 가지 분기가 완벽히 동작함을 확인했습니다.
2. **df 전체 적용 결과:** 전체 12건 중 `Fast`(3일 이하)가 5건, `Standard`(4일 이상)가 7건으로 분류되었습니다. 결측치 조건문(`pd.isna`)을 가장 먼저 분기 처리함으로써 `NaN`과 수치 비교 시 발생할 수 있는 잠재적 오류를 완벽히 차단했습니다.


## 문제 10. 고객 ID 분리 [선택 도전]

codes의 customer_id를 customer_type과 customer_number로 나누세요. 번호의 앞자리 0이 유지되게 하세요.

**점검 기준:** 3행×2열로 분리되며 customer_number는 "001" 같은 문자열입니다.

In [ ]:
# 문제 10 풀이
split_codes = codes['customer_id'].str.split('-', expand=True)
split_codes.columns = ['customer_type', 'customer_number']

codes_res = pd.concat([codes, split_codes], axis=1)
display(codes_res)
print("\ncustomer_number 자료형:", type(codes_res.loc[0, 'customer_number']))
print("앞자리 0 보존 확인   :", codes_res['customer_number'].tolist())


**결과 해석 / 확인한 점:**

1. `.str.split('-', expand=True)`를 사용하여 `CUST-001` 형태의 코드가 `customer_type`('CUST')과 `customer_number`('001')의 **3행 × 2열 DataFrame**으로 깔끔하게 분리되었습니다.
2. `customer_number`는 문자열(`str`) 타입을 유지하므로 '001', '002'처럼 앞자리의 0(leading zeros)이 숫자로 강제 변환되어 '1', '2'로 소실되지 않고 안전하게 보존되었습니다.


## 문제 11. 날짜 문자열 분리 [선택 도전]

date_text를 / 기준으로 month·day·year로 나누세요. 같은 자료를 format을 명시해 날짜로 변환하고 .dt.month와 비교하세요.

**점검 기준:** 두 번째 값은 2월 3일입니다. 문자열 "02"와 숫자 2의 차이를 설명하세요.

In [ ]:
# 문제 11 풀이
# 문자열 분리 방식
split_date = date_text.str.split('/', expand=True)
split_date.columns = ['month_str', 'day_str', 'year_str']

# datetime 변환 방식
dt_parsed = pd.to_datetime(date_text, format='%m/%d/%Y')

date_comp = pd.DataFrame({
    'date_text': date_text,
    'month_str': split_date['month_str'],
    'dt_month': dt_parsed.dt.month
})
display(date_comp)
print(f"\n두 번째 값 비교: month_str='{date_comp.loc[1, 'month_str']}', dt_month={date_comp.loc[1, 'dt_month']}")


**결과 해석 / 확인한 점:**

1. **두 번째 값 비교:** 두 번째 관측치 `02/03/2024`에서 문자열 분리는 `"02"`(문자열), `pd.to_datetime().dt.month`는 `2`(정수)를 반환합니다.
2. **문자열 `"02"`와 숫자 `2`의 차이:**
   - `"02"`는 텍스트 표현에 불과하여 월별 날짜 연산(예: 다음 달 계산, 분기 계산)이 불가능하며 서식에 종속적입니다.
   - 숫자 `2`는 실제 캘린더 상의 월(Month) 값을 나타내는 정수형 수치이므로 대소 비교나 분기 계산(`(month-1)//3 + 1`) 등에 자유롭게 사용할 수 있습니다.


## 문제 12. 매주 월요일 날짜 [선택 도전]

rrule로 2024-01-01부터 월요일 날짜 5개를 생성하고 YYYY-MM-DD로 출력하세요. 모든 날짜의 weekday가 0인지 확인하세요.

**점검 기준:** 1월 1·8·15·22·29일입니다. 주기와 시작일 포함 규칙을 적으세요.

**차근차근:** `from dateutil.rrule import rrule, WEEKLY, MO, FR`와 `from datetime import datetime`을 먼저 실행합니다. freq=WEEKLY는 매주, byweekday=MO/FR은 월/금요일, count는 생성 개수, dtstart는 시작일입니다. 시작일이 요일 조건에 맞으면 첫 날짜에 포함됩니다.

In [ ]:
# 문제 12 풀이
from datetime import datetime
from dateutil.rrule import rrule, WEEKLY, MO

# 2024-01-01부터 매주 월요일 5개 생성
start_dt = datetime(2024, 1, 1)
mondays = list(rrule(freq=WEEKLY, dtstart=start_dt, byweekday=MO, count=5))

monday_str = [d.strftime('%Y-%m-%d') for d in mondays]
all_mon = all(d.weekday() == 0 for d in mondays)

print("생성된 월요일 날짜 목록:", monday_str)
print("모든 날짜의 weekday == 0 (월요일) 확인:", all_mon)


**결과 해석 / 확인한 점:**

1. 생성된 5개의 날짜는 **`2024-01-01`, `2024-01-08`, `2024-01-15`, `2024-01-22`, `2024-01-29`**로 점검 기준과 정확히 일치합니다.
2. `d.weekday() == 0` 검증 결과 5개 날짜 모두 파이썬 `datetime` 모듈의 월요일 식별자(0)와 부합합니다.
3. **주기와 시작일 포함 규칙:** 시작일인 2024년 1월 1일이 이미 월요일이므로 `rrule`의 첫 번째 생성 원소에 즉시 포함되며, 이후 `freq=WEEKLY` 주기에 따라 정확히 7일 간격으로 5개 날짜가 생성됩니다.


## 문제 13. 매주 금요일과 대안 [선택 도전]

2024-03-01부터 금요일 4개를 rrule로 만드세요. date_range의 W-FRI로도 만들어 날짜 목록을 비교하세요.

**점검 기준:** 3월 1·8·15·22일입니다. list 변환 전 생성 개수를 제한하세요.

**차근차근:** `from dateutil.rrule import rrule, WEEKLY, MO, FR`와 `from datetime import datetime`을 먼저 실행합니다. freq=WEEKLY는 매주, byweekday=MO/FR은 월/금요일, count는 생성 개수, dtstart는 시작일입니다. 시작일이 요일 조건에 맞으면 첫 날짜에 포함됩니다.

In [ ]:
# 문제 13 풀이
from dateutil.rrule import FR

# rrule 방식: 2024-03-01부터 금요일 4개
fridays_rrule = [d.strftime('%Y-%m-%d') for d in rrule(freq=WEEKLY, dtstart=datetime(2024, 3, 1), byweekday=FR, count=4)]

# pd.date_range 방식: W-FRI 빈도
fridays_dr = pd.date_range(start='2024-03-01', periods=4, freq='W-FRI').strftime('%Y-%m-%d').tolist()

print("rrule 결과      :", fridays_rrule)
print("pd.date_range 결과:", fridays_dr)
print("두 결과 목록 일치 여부:", fridays_rrule == fridays_dr)


**결과 해석 / 확인한 점:**

1. 두 방식 모두 **`2024-03-01`, `2024-03-08`, `2024-03-15`, `2024-03-22`**의 동일한 4개 금요일 날짜를 생성했습니다.
2. **주의점:** `rrule`은 종료 조건(`until` 또는 `count`)을 지정하지 않으면 무한 제너레이터로 동작하므로 `list()`로 감싸기 전에 반드시 `count` 옵션으로 생성 개수를 제한해야 메모리 고갈을 방지할 수 있습니다.


## 문제 14. 통합 변환 보고서 [선택 도전]

원본 복사본에 할인율 백분율과 배송 분류를 추가하고 평점 오름차순·배송일수 내림차순·주문 ID 오름차순으로 정렬하세요. 앞 5행과 원본 보존 여부를 확인하세요.

**점검 기준:** 새 열을 더해도 원본 df와 df_original은 동일해야 합니다. 각 변환의 입력·출력 자료형을 기록하세요.

In [ ]:
# 문제 14 풀이
report_df = df.copy()

# 1. 할인율 백분율 추가
report_df['discount_percent'] = report_df['discount'] * 100

# 2. 배송 분류 추가
report_df['delivery_class'] = report_df['delivery_days'].apply(
    lambda d: 'Unknown' if pd.isna(d) else ('Fast' if d <= 3 else 'Standard')
)

# 3. 다중 정렬: 평점 오름차순, 배송일수 내림차순, 주문 ID 오름차순
sorted_final = report_df.sort_values(
    by=['customer_rating', 'delivery_days', 'order_id'],
    ascending=[True, False, True]
).reset_index(drop=True)

print("=== 최종 변환 및 정렬 보고서 앞 5행 ===")
cols_show = ['order_id', 'customer_rating', 'delivery_days', 'discount_percent', 'delivery_class', 'revenue']
display(sorted_final[cols_show].head(5))

# 원본 보존 여부 검증
is_original_safe = df.equals(df_original)
print(f"\n원본 df와 df_original 완벽 일치 여부: {is_original_safe}")


**결과 해석 / 확인한 점:**

### 1. 통합 변환 및 정렬 보고서 검증
- **변환 파이프라인:** `discount`(float) -> `discount_percent`(float, 100배), `delivery_days`(int) -> `delivery_class`(category/str, Fast/Standard)의 파생 변수가 정상 생성되었습니다.
- **다중 정렬 검증:** 최저 평점인 107번(2.0점, 8일), 111번(2.2점, 9일), 104번(2.5점, 7일), 109번(3.0점, 6일), 106번(3.5점, 5일) 순으로 우선순위 규칙에 따라 엄격하게 정렬되었습니다.

### 2. 원본 데이터 불변성 검증
- 파생 변수 생성과 정렬을 독립된 복사본 `report_df`에서 수행하였으므로 `df.equals(df_original)` 검증 결과 `True`를 반환하여 원본 데이터셋의 무결성이 완벽히 유지되었습니다.


## 선택 도전: 동점일 때 정렬 기준 정하기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_df = pd.DataFrame({'order_id':[1,2,3], 'revenue':[100,200,200]})
```

### 진행 순서

1. revenue는 큰 값부터, 같은 매출이면 order_id는 작은 값부터 정렬합니다.
2. order_id 목록을 출력합니다.
3. 두 정렬 기준에 대응하는 ascending 값 두 개를 말로 설명합니다.

**완료 기준:** order_id 순서 [2,3,1]. 매출 내림차순은 False, ID 오름차순은 True.

### 선택 도전 풀이

```python
extra_sorted=extra_df.sort_values(['revenue','order_id'],ascending=[False,True])
print(extra_sorted['order_id'].tolist())
assert extra_sorted['order_id'].tolist()==[2,3,1]
```

## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 모든 점검 기준에 대해 결과와 이유를 남겼나요?
- 행 수·고유 개수·합계·평균의 분모를 구분했나요?
- 원본과 임시 변경본을 구분했나요?
- 결측·중복·제외 조건과 추가 확인할 사항을 기록했나요?